# POTER Waterbirds — Kaggle Reproduction Notebook

Select **GPU T4 x2** and enable Internet in Kaggle settings. Run cells sequentially. Training uses batch size 128, then trains for 300 epochs per seed on a single GPU.

Checkpoints are committed after each epoch to `/kaggle/working/poter-results`. Before exiting, run the backup cell and download the resulting zip archive. In a new session, restore that backup into `/kaggle/working/poter-results` before continuing.


In [ ]:
import base64, io, zipfile, pathlib, os, sys, subprocess
ROOT = pathlib.Path('/kaggle/working/POTER')
ROOT.mkdir(parents=True, exist_ok=True)
payload = "..."
with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as z:
    z.extractall(ROOT)
os.chdir(ROOT)
import torch, torchvision, numpy, pandas, scipy, sklearn, tqdm, PIL
print('Torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable GPU T4 x2 in Settings > Accelerator.'
print('GPU:', torch.cuda.get_device_name(0), 'VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


## 1. Download Waterbirds and extract features
Source dataset from Stanford NLP. Requires Internet connection (~490 MB archive).


In [ ]:
import urllib.request, tarfile, hashlib
DATA = pathlib.Path('/kaggle/working/poter-data')
FEATURE = pathlib.Path('/kaggle/working/poter-features/waterbirds_standard.pt')
archive = pathlib.Path('/kaggle/working/waterbirds.tar.gz')
dest = DATA / 'cub/data'
meta = dest / 'waterbird_complete95_forest2water2/metadata.csv'
if not meta.exists():
    if not archive.exists():
        urllib.request.urlretrieve('https://downloads.cs.stanford.edu/nlp/data/dro/waterbird_complete95_forest2water2.tar.gz', archive)
    assert hashlib.sha256(archive.read_bytes()).hexdigest() == '56c51b77f7d17283ac97003d973e9107e8af9ceebffcb0c8be14d024f846f951', 'Archive checksum mismatch'
    dest.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, 'r:gz') as tar:
        for m in tar.getmembers():
            assert (dest / m.name).resolve().is_relative_to(dest.resolve()) and (m.isfile() or m.isdir())
        tar.extractall(dest)
assert meta.exists()
FEATURE.parent.mkdir(parents=True, exist_ok=True)
if not FEATURE.exists():
    subprocess.run([sys.executable, 'extract_features.py', '--dataset','CUB','--data_dir',str(DATA),'--ref_per_group','0:0,1:466,2:133,3:0','--ref_seed','0','--output',str(FEATURE),'--batch_size','32','--gpu','0'], check=True)
print('Dataset:', meta, 'Cache:', FEATURE)


## 2. Probe run with batch size 128 (2 epochs)
Measures execution time and VRAM usage to verify batch 128 fits comfortably on a Tesla T4.


In [ ]:
RESULTS = pathlib.Path('/kaggle/working/poter-results')
probe = RESULTS / 'probe_batch128'
subprocess.run([sys.executable, 'scripts/kaggle_waterbirds.py', '--data-dir',str(DATA),'--cache',str(FEATURE),'--run-dir',str(probe),'--seed','1','--epochs','2','--batch-size','128','--max-hours','1'], check=True)


## 3. Train 300 epochs per seed
Change `SEED` to 1, 2, or 3. Each run automatically resumes from the saved checkpoint if available. Near session timeout, it commits the current epoch checkpoint and exits gracefully.


In [ ]:
SEED = 1  # change to 2, 3 after previous seed completes
run_dir = RESULTS / 'CUB' / 'Clean' / f'POTER_CUB_seed{SEED}_kaggle'
subprocess.run([sys.executable, 'scripts/kaggle_waterbirds.py', '--data-dir',str(DATA),'--cache',str(FEATURE),'--run-dir',str(run_dir),'--seed',str(SEED),'--epochs','300','--batch-size','128','--max-hours','9.5'], check=True)


## Save checkpoint archive before session ends
Download the zip archive from the Kaggle Output panel. For a new session, upload the zip to Input and extract into `/kaggle/working/poter-results` before continuing.


In [ ]:
import shutil
backup = shutil.make_archive('/kaggle/working/poter-results-backup', 'zip', RESULTS)
print('Download this archive from Output:', backup)


## 4. Summarize benchmark results
Selects epoch by validation Worst-Group Accuracy (WGA), then evaluates held-out test split at that exact epoch.


In [ ]:
# Summarize single seed or all 3 seeds:
subprocess.run([sys.executable, 'scripts/summarize_waterbirds.py', str(RESULTS), '--seeds', str(SEED)], check=True)
